In [ ]:
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_PATH, PDB_DIR = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/", "/mnt/volume6/czj/labLGN/LabLZ/models/alphafold_pdb/"
os.makedirs(PDB_DIR, exist_ok=True)
uniprot_ids = pd.read_csv(BASE_PATH + "cell2024_final.csv")["Uniprot"].dropna().unique()

In [2]:
def fetch(uid):
    out = PDB_DIR + f"{uid}.pdb"
    if os.path.exists(out):
        return None
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v6.pdb"
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            with open(out, "w") as f: f.write(r.text)
            return None
        return (uid, r.status_code)
    except Exception as e:
        return (uid, str(e))

failed = []
with ThreadPoolExecutor(max_workers=12) as ex:
    futs = [ex.submit(fetch, u) for u in uniprot_ids]
    for i, fut in enumerate(as_completed(futs)):
        r = fut.result()
        if r: failed.append(r)
        if i % 50 == 0: print(f"Progress: {i}/{len(uniprot_ids)}, failed {len(failed)}")

pd.DataFrame(failed, columns=["Uniprot", "Status"]).to_csv(BASE_PATH + "alphafold_failed.csv", index=False)
print(f"Finished. Failed: {len(failed)}")

Progress: 0/915, failed 0
Progress: 50/915, failed 0
Progress: 100/915, failed 0
Progress: 150/915, failed 0
Progress: 200/915, failed 0
Progress: 250/915, failed 0
Progress: 300/915, failed 0
Progress: 350/915, failed 0
Progress: 400/915, failed 0
Progress: 450/915, failed 0
Progress: 500/915, failed 0
Progress: 550/915, failed 0
Progress: 600/915, failed 0
Progress: 650/915, failed 0
Progress: 700/915, failed 0
Progress: 750/915, failed 0
Progress: 800/915, failed 0
Progress: 850/915, failed 0
Progress: 900/915, failed 0
Finished. Failed: 5


In [ ]:
import os
import re
import warnings
import pandas as pd
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley

import biotite.structure.io.pdb as pdb_io
import biotite.structure as struc

warnings.filterwarnings("ignore")


BASE_PATH  = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"
PDB_DIR    = "/mnt/volume6/czj/labLGN/LabLZ/models/alphafold_pdb/"
INPUT_CSV  = BASE_PATH + "cell2024_final_with_labels.csv"
OUTPUT_CSV = BASE_PATH + "phase35_struct_features.csv"
SAVE_EVERY = 100

TEST_MODE = False    # Test mode: only handle small subset of data, write to separate file
TEST_N    = 20

if TEST_MODE:
    OUTPUT_CSV = OUTPUT_CSV.replace(".csv", "_TEST.csv")
    SAVE_EVERY = 5


# Kyte-Doolittle hydrophobicity scale
KD_SCALE = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2,
}

# Tien et al. 2013 maximum ASA
MAX_ASA = {
    'A': 129.0, 'R': 274.0, 'N': 195.0, 'D': 193.0, 'C': 167.0,
    'Q': 225.0, 'E': 223.0, 'G': 104.0, 'H': 224.0, 'I': 197.0,
    'L': 201.0, 'K': 236.0, 'M': 224.0, 'F': 240.0, 'P': 159.0,
    'S': 155.0, 'T': 172.0, 'W': 285.0, 'Y': 263.0, 'V': 174.0,
}

# SAFETY: three aa to one aa mapping, defensive
THREE_TO_ONE = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
    'MSE': 'M',
}

def three_to_one(resname):
    return THREE_TO_ONE.get(resname.strip().upper(), "X")

def parse_mutation(m):
    if not isinstance(m, str): return None, None, None
    mt = re.match(r'^([A-Z])(\d+)([A-Z])$', m.strip())
    return (mt.group(1), int(mt.group(2)), mt.group(3)) if mt else (None, None, None)

def compute_protein(uid):
    # AF PDB → {pos: (res_aa, plddt, sasa, ss)}
    path = PDB_DIR + f"{uid}.pdb"
    if not os.path.exists(path): 
        return uid, None
    try:
        structure = PDBParser(QUIET=True).get_structure(uid, path)
        ShrakeRupley().compute(structure, level="R")
        chain = list(structure[0].get_chains())[0]
        # SSE
        try:
            arr = pdb_io.PDBFile.read(path).get_structure(model=1)
            arr = arr[struc.filter_amino_acids(arr)]
            arr = arr[arr.chain_id == arr.chain_id[0]]
            m = {"a":"H","b":"E","c":"C"}
            sse = {int(r): m.get(s,"C") for r,s in zip(struc.get_residues(arr)[0], struc.annotate_sse(arr))}
        except Exception:
            sse = {}
        out = {}
        for res in chain.get_residues():
            rid = res.get_id()
            if rid[0] != " ": continue
            ca = [a for a in res.get_atoms() if a.get_name()=="CA"]
            plddt = (ca[0] if ca else next(iter(res.get_atoms()))).get_bfactor()
            out[rid[1]] = (three_to_one(res.get_resname()), plddt, getattr(res,"sasa",np.nan), sse.get(rid[1]))
        return uid, out
    except Exception as e:
        return uid, f"error:{str(e)[:50]}"


df = pd.read_csv(INPUT_CSV)
df_eval = df[df["mutant_sequence"].notna() & df["Mislocalized"].notna()].copy()
uids = [u for u in df_eval["Uniprot"].dropna().unique()]
print(f"unique proteins: {len(uids)}, rows: {len(df_eval)}")

prot = {}
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for uid, feats in tqdm(ex.map(compute_protein, uids, chunksize=4), total=len(uids), desc="Computing protein features"):
        prot[uid] = feats

# Assemble results
rows = []
for _, row in df_eval.iterrows():
    uid = row.get("Uniprot")
    wt, pos, mt = parse_mutation(row.get("Mutation_used"))
    r = {"plddt":np.nan,"sasa":np.nan,"rsa":np.nan,"delta_hydrophobicity":np.nan,
         "ss_type":"unknown","ss_helix":np.nan,"ss_strand":np.nan,"ss_coil":np.nan,"struct_status":"ok"}
    if wt in KD_SCALE and mt in KD_SCALE:
        r["delta_hydrophobicity"] = KD_SCALE[mt] - KD_SCALE[wt]
    pf = prot.get(uid)
    if wt is None or pd.isna(uid):      r["struct_status"]="no_mutation_or_uniprot"
    elif pf is None:                    r["struct_status"]="no_pdb"
    elif isinstance(pf, str):           r["struct_status"]=pf
    elif pos not in pf:                 r["struct_status"]="pos_not_found"
    else:
        res_aa, plddt, sasa, ss = pf[pos]
        if res_aa != wt:                # Safety check: AF structure residue mismatch with WT
            r["struct_status"]=f"wt_mismatch(struct={res_aa},label={wt})"
        else:
            r["plddt"], r["sasa"] = plddt, sasa
            if not np.isnan(sasa) and wt in MAX_ASA: r["rsa"]=min(sasa/MAX_ASA[wt],1.0)
            if ss:  r.update(ss_type=ss, ss_helix=int(ss=="H"), ss_strand=int(ss=="E"), ss_coil=int(ss=="C"))
    rows.append({"Variant":row.get("Variant"),"Gene":row.get("Gene"),"Uniprot":uid,
                 "Mutation_used":row.get("Mutation_used"),"Mislocalized":row.get("Mislocalized"), **r})

pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
print("done", pd.DataFrame(rows)["struct_status"].value_counts().to_dict())


unique proteins: 871, rows: 2179


Computing protein features: 100%|██████████| 871/871 [00:08<00:00, 107.97it/s]

done {'ok': 2168, 'no_pdb': 11}


查看数据

In [5]:
df_out = pd.read_csv(OUTPUT_CSV)              # 正常读，NaN 保持 NaN
print(f"Rows: {len(df_out)}")

print("\nstruct_status distribution:")
print(df_out["struct_status"].value_counts())

print("\nss_type distribution (H/E/C/unknown):")
print(df_out["ss_type"].value_counts())

for col in ["plddt", "sasa", "rsa", "delta_hydrophobicity"]:
    print(f"{col} not null: {df_out[col].notna().sum()}")

print("\nRange of numerical features:")
for col in ["plddt", "sasa", "rsa", "delta_hydrophobicity"]:
    s = df_out[col]
    print(f"{col:22s} min={s.min():.2f}  median={s.median():.2f}  max={s.max():.2f}")

Rows: 2179

struct_status distribution:
struct_status
ok        2168
no_pdb      11
Name: count, dtype: int64

ss_type distribution (H/E/C/unknown):
ss_type
C          1063
H           735
E           370
unknown      11
Name: count, dtype: int64
plddt not null: 2168
sasa not null: 2168
rsa not null: 2168
delta_hydrophobicity not null: 2179

Range of numerical features:
plddt                  min=24.28  median=94.62  max=98.94
sasa                   min=0.00  median=41.56  max=245.85
rsa                    min=0.00  median=0.22  max=0.92
delta_hydrophobicity   min=-9.00  median=0.00  max=9.00


# Merge

In [ ]:
import pandas as pd

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"

df_main = pd.read_csv(BASE_PATH + "cell2024_final_with_labels.csv")
df_feat = pd.read_csv(BASE_PATH + "phase35_struct_features.csv")

# 2. Use the full Variant string as the join key 
feature_cols = [
    "plddt", "sasa", "rsa",
    "ss_type", "ss_helix", "ss_strand", "ss_coil",
    "delta_hydrophobicity",
    "struct_status",
]
df_feat_slim = df_feat[["Gene", "Variant"] + feature_cols].copy()

# 3. Verify Variant is unique in the feature table
dup = df_feat_slim[["Gene", "Variant"]].duplicated().sum()
if dup:
    print(f"WARNING: {dup} duplicate Variant(s) in the feature table — please investigate (keeping first).")
    df_feat_slim = df_feat_slim.drop_duplicates(["Gene", "Variant"], keep="first")

# 4. Left join: enforces uniqueness on the feature side.
df_merged = df_main.merge(df_feat_slim, on=["Gene", "Variant"], how="left", validate="m:1")

# 5. Sanity-check the merge
print(f"Main rows: {len(df_main)}, after merge: {len(df_merged)}")
print(f"Rows with structural features: {df_merged['plddt'].notna().sum()}")
print(f"Rows without matched features: {df_merged['struct_status'].isna().sum()}")

# 6. Merge
df_model = df_merged[
    df_merged["Mislocalized"].notna()
    & df_merged["mutant_sequence"].notna()
].copy()

print(f"\nSingle-substitution training set: {len(df_model)} rows")
print(df_model["Mislocalized"].value_counts())
print(f"Rows with structural features: {df_model['plddt'].notna().sum()}")

pos = (df_model["Mislocalized"] == 1).sum()
neg = (df_model["Mislocalized"] == 0).sum()
print(f"scale_pos_weight ~ {neg / pos:.2f}")

df_merged.to_csv(BASE_PATH + "cell2024_with_struct_features.csv", index=False)
df_model.to_csv(BASE_PATH + "cell2024_model_single_subst.csv", index=False)

Main rows: 2179, after merge: 2179
Rows with structural features: 2168
Rows without matched features: 0

Single-substitution training set: 2179 rows
Mislocalized
0.0    1943
1.0     236
Name: count, dtype: int64
Rows with structural features: 2168
scale_pos_weight ~ 8.23
